# 02 - Scanned PDF NER

Pipeline: PDF -> PyMuPDF -> GLiNER -> predicted entities -> strict/relaxed evaluation.

In [ ]:
import importlib
import ner.evaluation, ner.inference, ner.pdf, ner.utils

# reload so edits in ner/ are picked up without restarting the kernel
for module in (ner.utils, ner.pdf, ner.inference, ner.evaluation):
    importlib.reload(module)

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl


In [47]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_scanned'
OUTPUT_PATH = '../results/pdf_scanned/gliner_small_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [54]:
gold_samples = load_jsonl(VALIDATION_PATH)
# only 10 scanned pdfs exist, records without a pdf are skipped
pdf_samples = extract_pdf_dataset(gold_samples, PDF_DIR, suffix='_scanned')
print(len(pdf_samples), 'PDFs;', sum(1 for s in pdf_samples if s['text'].strip()), 'with text')
model = load_gliner_model()
responses = predict_dataset(model, pdf_samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)


TypeError: extract_pdf_dataset() got an unexpected keyword argument 'suffix'

In [ ]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)
